In [1]:
import pandas as pd
import json
from pathlib import Path
import csv
import numpy as np

In [2]:
book = '/kaggle/input/casml-dataset/Dataset_RAG (1)/book.pdf'    
queries = '/kaggle/input/casml-dataset/Dataset_RAG (1)/queries.json' 
output = Path('submission.csv')

In [3]:
import os

In [4]:
with open(queries, 'r', encoding='utf-8') as f:
    queries = json.load(f)

In [52]:
queries

[{'query_id': '1', 'question': 'What is the scientific method in psychology?'},
 {'query_id': '2', 'question': 'What are the basic parts of a neuron?'},
 {'query_id': '3', 'question': 'What are the stages of sleep?'},
 {'query_id': '4', 'question': 'What is operant conditioning?'},
 {'query_id': '5', 'question': 'What is problem-solving in psychology?'},
 {'query_id': '6', 'question': 'What are the three stages of memory?'},
 {'query_id': '7', 'question': 'What are the key components of emotion?'},
 {'query_id': '8',
  'question': 'What are the major personality traits in the Five Factor Model?'},
 {'query_id': '9', 'question': 'What is social psychology?'},
 {'query_id': '10', 'question': 'What is the sociocultural model in therapy?'},
 {'query_id': '11', 'question': 'What is the history of psychology?'},
 {'query_id': '12', 'question': 'Who were Wilhelm Wundt and William James?'},
 {'query_id': '13', 'question': 'What is functionalism in psychology?'},
 {'query_id': '14',
  'question

In [5]:
!pip install pdfplumber


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 65.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.1 MB/s eta 0:00:00


In [6]:
import pdfplumber

In [7]:
pages = []  
with pdfplumber.open(book) as pdf:
    for i, page in enumerate(pdf.pages, start=1):
        txt = page.extract_text(x_tolerance=2, y_tolerance=2) or ""
        txt = "\n".join([ln.strip() for ln in txt.splitlines() if ln.strip()])
        pages.append({"page_no": i, "text": txt})

In [8]:
import re

In [9]:
section_for_page = {}  # page_no -> section_title (if found)
current_section = None

section_heading_patterns = [
    r'^\s*CHAPTER\s+\d+', r'^\s*Chapter\s+\d+'
]

for p in pages:
    lines = p['text'].splitlines()
    found = None
    for ln in lines[:6]:
        if any(re.search(pat, ln, flags=re.IGNORECASE) for pat in section_heading_patterns):
            found = ln.strip()
            break
    # fallback: если нет "Chapter", то если первая строка короткая и выглядит как заголовок (есть буквы, не предложение)
    if not found and lines:
        first = lines[0].strip()
        if 3 <= len(first) <= 120 and (first.isupper() or (len(first.split())<=6 and first[0].isupper())):
            found = first
    if found:
        current_section = found
    section_for_page[p['page_no']] = current_section or "Unknown"


In [12]:
section_for_page

{1: 'Unknown',
 2: 'Unknown',
 3: 'Psychology 2e',
 4: 'OpenStax',
 5: 'OPENSTAX',
 6: 'Study where you want, what',
 7: 'CHAPTER 1',
 8: 'CHAPTER 1',
 9: 'CHAPTER 8',
 10: 'CHAPTER 8',
 11: 'CHAPTER 15',
 12: 'Access for free at openstax.org',
 13: 'Preface 1',
 14: 'Preface 1',
 15: 'Preface 3',
 16: 'Preface 3',
 17: 'Preface 5',
 18: 'Preface 5',
 19: 'Preface 5',
 20: 'Preface 5',
 21: 'Preface 5',
 22: 'Preface 5',
 23: 'Preface 5',
 24: 'Preface 5',
 25: 'Preface 5',
 26: 'Preface 5',
 27: 'Preface 5',
 28: 'Preface 5',
 29: 'Preface 5',
 30: 'Preface 5',
 31: 'Preface 5',
 32: 'Preface 5',
 33: 'Preface 5',
 34: 'Preface 5',
 35: 'Preface 5',
 36: 'Preface 5',
 37: 'Preface 5',
 38: 'Preface 5',
 39: 'Preface 5',
 40: 'Preface 5',
 41: 'Preface 5',
 42: 'Preface 5',
 43: 'Preface 5',
 44: 'Preface 5',
 45: 'Preface 5',
 46: 'Preface 5',
 47: 'Preface 5',
 48: 'Preface 5',
 49: 'Preface 5',
 50: 'Preface 5',
 51: 'Preface 5',
 52: 'Preface 5',
 53: 'Preface 5',
 54: 'Preface 5',

In [10]:
from sentence_transformers import SentenceTransformer

2025-11-11 13:59:16.746951: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762869556.929849      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762869556.998839      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [11]:
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
!pip install -q semantic-text-splitter nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 60.1 MB/s eta 0:00:0000:0100:01


In [13]:
import nltk
from semantic_text_splitter import TextSplitter

In [14]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [15]:
def chunk_page_text(page_text, page_no, section, max_chars=1200, overlap=200):
    chunks = []
    txt = page_text
    if not txt:
        return []
    i = 0
    L = len(txt)
    while i < L:
        part = txt[i: i + max_chars]
        chunks.append({
            "page_no": page_no,
            "section": section,
            "text": part
        })
        i += max_chars - overlap
    return chunks

chunks = []
for p in pages:
    chunks.extend(chunk_page_text(p['text'], p['page_no'], section_for_page[p['page_no']], max_chars=1200, overlap=200))

In [ ]:
#semantic chunks 
# ------------------------------
# Semantic chunking (sentence-aware)
# ------------------------------
# Требует только nltk (поставь если нужно: !pip install -q nltk)
import nltk
nltk.download('punkt', quiet=True)
from nltk.tokenize import sent_tokenize

def semantic_split_text(text, max_chars=1200, overlap=200):
    """
    Разбивает текст на семантические чанки:
      - сначала по абзацам (по двойному переводу строки),
      - затем внутри абзацев — по предложениям (nltk.sent_tokenize).
    Формирует чанки длиной <= max_chars символов, с перекрытием overlap.
    Возвращает список строк (чанков).
    """
    if not text or not text.strip():
        return []

    # 1) разбиваем на параграфы (или на строки, если параграфов нет)
    paras = [p.strip() for p in text.split('\n\n') if p.strip()]
    if len(paras) == 0:
        paras = [ln.strip() for ln in text.splitlines() if ln.strip()]
        if len(paras) == 0:
            paras = [text.strip()]

    chunks = []
    for para in paras:
        # 2) разбиваем параграф на предложения
        try:
            sents = sent_tokenize(para)
        except Exception:
            # fallback: если tokenization не сработала, делим по точкам
            sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', para) if s.strip()]

        # 3) собираем предложения в чанки, следя за max_chars
        cur = []
        cur_len = 0
        for sent in sents:
            sent_len = len(sent) + 1  # +1 for space
            # если текущее + предложение в пределах max_chars — добавляем
            if cur_len + sent_len <= max_chars or cur_len == 0:
                cur.append(sent)
                cur_len += sent_len
            else:
                # flush current chunk
                chunk_text = " ".join(cur).strip()
                if chunk_text:
                    chunks.append(chunk_text)
                # start new chunk with this sentence
                cur = [sent]
                cur_len = sent_len
        # flush tail
        if cur:
            chunk_text = " ".join(cur).strip()
            if chunk_text:
                chunks.append(chunk_text)

    # 4) apply character overlap: create overlapped list
    if overlap <= 0 or len(chunks) <= 1:
        return chunks

    overlapped = []
    for i, c in enumerate(chunks):
        # add current chunk
        overlapped.append(c)
        # create overlap with next chunk by appending tail chars from current to next
        if i + 1 < len(chunks):
            # take last `overlap` chars of current and prepend to next if not present
            tail = c[-overlap:]
            # if next chunk already starts with that tail, skip
            nxt = chunks[i+1]
            if not nxt.startswith(tail):
                overlapped.append(tail + " " + nxt)
    # Remove possible duplicates while keeping order
    seen = set()
    final = []
    for ch in overlapped:
        ch_norm = ch.strip()
        if ch_norm not in seen:
            seen.add(ch_norm)
            final.append(ch_norm)
    return final

# ------------------------------
# Пример интеграции: строим chunks с метаданными
# ------------------------------
chunks = []   # итог: список dicts {page_no, section, text}
for p in pages:
    page_no = p['page_no']
    section = section_for_page.get(page_no, "Unknown")
    text = p['text']
    # семантическая нарезка страницы
    sem_chunks = semantic_split_text(text, max_chars=1200, overlap=200)
    for ch in sem_chunks:
        chunks.append({
            "page_no": page_no,
            "section": section,
            "text": ch
        })

print("Semantic chunks:", len(chunks))
# Далее используем chunks как раньше: texts = [c['text'] for c in chunks] -> embeddings -> FAISS etc.


In [16]:
len(chunks)

2642

In [17]:
import tqdm

In [66]:
semantic_chunks[10]

{'page_no': 7,
 'text': '36\n2.2 Approaches to Research 41\n2.3 Analyzing Findings 48\n2.4 Ethics 59\nKey Terms 63\nSummary 64\nReview Questions 66\nCritical Thinking Questions 69\nPersonal Application Questions 70\nCHAPTER 3\nBiopsychology\n71\nIntroduction 71\n3.1 Human Genetics 72\n3.2 Cells of the Nervous System 78\n3.3 Parts of the Nervous System 84\n3.4 The Brain and Spinal Cord 86\n3.5 The Endocrine System 97\nKey Terms 100\nSummary 102\nReview Questions 103\nCritical Thinking Questions 106\nPersonal Application Questions 106\nCHAPTER 4\nStates of Consciousness\n109\nIntroduction 109\n4.1 What Is Consciousness? 110',
 'section': 'CHAPTER 1'}

In [19]:
texts = [c['text'] for c in chunks]
batch = 128
emb_list = []
for i in tqdm.tqdm(range(0, len(texts), batch), desc="Embedding"):
    sub = texts[i:i+batch]
    em = embedder.encode(sub, show_progress_bar=False, normalize_embeddings=True)
    emb_list.append(em)
embeddings = np.vstack(emb_list).astype('float32')  # shape = (N, dim)
print(embeddings.shape)

Embedding: 100%|██████████| 21/21 [00:06<00:00,  3.41it/s]

(2642, 384)


In [126]:
embeddings[0]

array([ 5.80102615e-02,  6.18371554e-02,  3.87069932e-03,  9.48799998e-02,
       -5.69940135e-02,  9.56900534e-04,  4.33432423e-02,  1.12137692e-02,
        1.02925606e-01, -3.24500985e-02,  1.30822838e-04,  1.64513884e-03,
        6.59969449e-03,  1.65128596e-02,  4.31401357e-02, -1.51521591e-02,
        5.27998172e-02,  3.99324065e-03, -1.09881893e-01, -1.25417775e-02,
       -8.35519563e-03, -3.02352272e-02,  4.78644297e-02, -2.77162194e-02,
       -1.00386046e-01,  1.33872023e-02, -2.81945970e-02, -1.25867516e-01,
        1.19231716e-02, -4.79624560e-03, -4.69934754e-03, -5.83288609e-04,
        3.63251567e-02,  5.62651735e-03, -4.91327345e-02, -1.18629681e-02,
       -1.91735197e-02,  1.13749385e-01, -4.96260356e-03,  3.33071133e-04,
       -6.32092282e-02, -8.91803764e-03,  5.92968520e-03, -3.34820636e-02,
       -1.25231994e-02, -7.33031631e-02, -2.82205753e-02, -3.48970406e-02,
       -1.06925905e-01, -5.22217304e-02, -8.34634379e-02, -1.30079091e-02,
       -9.68262646e-03,  

In [20]:
!pip install faiss-cpu

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 57.0 MB/s eta 0:00:00:00:0100:01


In [21]:
import faiss

In [22]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   
index.add(embeddings)

In [128]:
index

<faiss.swigfaiss_avx512.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x7c52d0b19020> >

In [129]:
embeddings.shape

(2642, 384)

In [23]:
print(index.ntotal)

2642


In [24]:
meta = [{"page_no": c["page_no"], "section": c["section"], "text": c["text"]} for c in chunks]

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [26]:
gen_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

In [27]:
tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
model = AutoModelForCausalLM.from_pretrained(
    gen_model_name,
    torch_dtype="auto")

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0,  
    do_sample=False,
    max_new_tokens=400
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [28]:
def build_prompt(question, contexts):
    ctx_text = ""
    for i,c in enumerate(contexts, start=1):
        ctx_text += f"[CONTEXT {i}] SECTION: {c['section']} | PAGE: {c['page_no']}\n{c['text']}\n\n"
    prompt = (
        "You are an expert psychology assistant. "
        "Answer ONLY based on the provided textbook context. "
        "If the information is missing, say 'No data found in the textbook.'\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{ctx_text}\n\n"
        "Give a concise answer in English. "
        "Then output JSON with fields 'sections' and 'pages' for the cited sources.\n"
        "Format example: {\"sections\": [\"...\"], \"pages\": [\"...\"]}\n"
    )
    return prompt

In [133]:
def retrieve(query, top_k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    D, I = index.search(q_emb, top_k)
    idxs = I[0].tolist()
    results = [meta[i] for i in idxs]
    return results

In [ ]:
'What is the scientific method in psychology?'

In [93]:
results, q_emb, D, I, idx = retrieve('What is the scientific method in psychology?', top_k=5)

In [99]:
idx

[52, 54, 219, 164, 172]

In [95]:
q_emb

array([[ 3.31251360e-02,  7.72634894e-02, -6.05272129e-02,
         5.58931977e-02, -4.29528691e-02,  3.40702012e-02,
        -4.68931273e-02,  8.37683529e-02,  1.73148196e-02,
         5.77760600e-02, -4.08849586e-03, -3.55258060e-04,
        -3.69375013e-02,  4.95445356e-02, -5.59994057e-02,
        -1.49594778e-02, -4.08309661e-02,  1.36495447e-02,
         4.11540568e-02, -4.58180085e-02,  1.56285353e-02,
        -4.89531159e-02,  4.34310250e-02, -1.45325204e-02,
        -3.63519415e-02,  6.56595966e-03, -3.96012291e-02,
        -5.97417057e-02,  6.11775927e-03,  1.51607729e-02,
         8.40146020e-02,  6.84476718e-02,  1.99613273e-02,
         1.33484667e-02, -1.23243809e-01,  6.92226961e-02,
        -5.47430553e-02,  7.71259964e-02,  6.03782199e-02,
         7.61651397e-02, -7.55049214e-02, -1.05087794e-02,
         5.04769199e-03, -3.72153637e-03,  2.40645241e-02,
        -4.30079289e-02, -3.55206169e-02, -1.39321201e-02,
        -2.73424909e-02, -1.63208935e-02, -1.06247388e-0

In [94]:
results

[{'page_no': 24,
  'section': 'Preface 5',
  'text': 'h a sensory experience can be broken down into individual parts, how those parts relate to each other\nas a whole is often what the individual responds to in perception. For example, a song may be made up of\nindividual notes played by different instruments, but the real nature of the song is perceived in the\ncombinations of these notes as they form the melody, rhythm, and harmony. In many ways, this particular\nperspective would have directly contradicted Wundt’s ideas of structuralism (Thorne & Henley, 2005).\nUnfortunately, in moving to the United States, these scientists were forced to abandon much of their work and\nwere unable to continue to conduct research on a large scale. These factors along with the rise of behaviorism\n(described next) in the United States prevented principles of Gestalt psychology from being as influential in\nthe United States as they had been in their native Germany (Thorne & Henley, 2005). Despite t

In [29]:
!pip install -q sentence-transformers accelerate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.3 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installe

In [30]:
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
import numpy as np
import random
from tqdm.auto import tqdm

In [32]:
# Можно дообучить (если есть пары)
model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
reranker = CrossEncoder(model_name, num_labels=1)
# или: reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_candidates(query, candidates, reranker, batch_size=32):
    pairs = [(query, c['text']) for c in candidates]
    scores = reranker.predict(pairs, batch_size=batch_size, show_progress_bar=False)
    for c, s in zip(candidates, scores):
        c['score_rerank'] = float(s)
    return sorted(candidates, key=lambda x: x['score_rerank'], reverse=True)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [40]:
import os
import re
import json
import time
import gc
import torch
from tqdm.auto import tqdm

# Убрать лишние сообщения от transformers
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
# Помогает уменьшить фрагментацию GPU-аллокатора
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ----- Параметры -----
TOP_K_RETRIEVE = 30
TOP_AFTER_RERANK = 5
MAX_NEW_TOKENS = 300
RERANK_BATCH = 8
CLEAR_EVERY = 2

# ---------- Утилиты ----------

def safe_extract_generated_text(gen_out):
    """
    Универсально извлекает текст из разных форматов вывода pipeline.
    Поддерживает:
     - [{'generated_text': "...."}]
     - [{'generated_text': [{'role':'assistant','content':'...'}]}]
     - [{'generated_text': "..."}, ...]
     - str
    """
    # list
    if isinstance(gen_out, list) and len(gen_out) > 0:
        first = gen_out[0]
        # case: {'generated_text': ...}
        if isinstance(first, dict) and 'generated_text' in first:
            v = first['generated_text']
            # case nested chat messages list
            if isinstance(v, list) and len(v) > 0:
                last = v[-1]
                if isinstance(last, dict) and 'content' in last:
                    return last['content']
                if isinstance(last, str):
                    return last
                # fallback join any strings inside
                return " ".join([x.get('content', str(x)) if isinstance(x, dict) else str(x) for x in v])
            # case plain string
            if isinstance(v, str):
                return v
        # case first is string
        if isinstance(first, str):
            return first
    # dict with generated_text
    if isinstance(gen_out, dict) and 'generated_text' in gen_out:
        v = gen_out['generated_text']
        if isinstance(v, str):
            return v
        if isinstance(v, list) and len(v) > 0:
            last = v[-1]
            if isinstance(last, dict) and 'content' in last:
                return last['content']
            return str(last)
    # fallback
    return str(gen_out)

def call_generator(messages_or_prompt, **gen_kwargs):
    """
    Wrapper to call `generator` safely and return a single string output.
    messages_or_prompt: either a prompt string or messages list (for chat-like pipelines).
    gen_kwargs: allowed generation args (we filter unsupported keys).
    """
    # Filter out unsupported args: temperature/top_p/top_k often cause warnings in some pipelines
    allowed = {"max_new_tokens", "num_return_sequences", "do_sample", "top_k", "top_p", "temperature"}
    # But we will NOT pass temperature/top_p/top_k to avoid warnings: keep only safe keys
    safe_keys = {"max_new_tokens", "num_return_sequences", "do_sample"}
    call_args = {k: v for k, v in gen_kwargs.items() if k in safe_keys}

    # Call generator
    out = generator(messages_or_prompt, **call_args)
    return safe_extract_generated_text(out)


# Защищённый retrieve: не выходит за границы meta
def retrieve(query, top_k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype('float32')
    k = min(top_k, max(1, getattr(index, "ntotal", 0)))  # safe
    if k == 0:
        return [], None, None, [], []
    D, I = index.search(q_emb, k)
    idxs = [int(x) for x in I[0].tolist()]
    # фильтруем индексы, которые реально в meta
    valid_idxs = [i for i in idxs if 0 <= i < len(meta)]
    results = [meta[i] for i in valid_idxs]
    return results, q_emb, D, I, valid_idxs

# Устойчивый reranker
def rerank_candidates(query, candidates, reranker, batch_size=32):
    """
    candidates - list of dicts OR list of strings.
    Возвращает список словарей: {'text':..., 'page_no':..., 'section':..., 'score_rerank': ...}
    """
    standardized = []
    for c in candidates:
        if isinstance(c, dict):
            standardized.append(c.copy())
        else:
            # assume string -> wrap into dict
            standardized.append({"text": str(c), "page_no": None, "section": None})

    # prepare pairs
    pairs = [(query, c['text']) for c in standardized]
    if len(pairs) == 0:
        return standardized

    # predict scores (batching внутри CrossEncoder.predict)
    scores = reranker.predict(pairs, batch_size=batch_size, show_progress_bar=False)
    for c, s in zip(standardized, scores):
        c['score_rerank'] = float(s)
    standardized_sorted = sorted(standardized, key=lambda x: x.get('score_rerank', 0.0), reverse=True)
    return standardized_sorted

# flatten recursive lists of context into list of strings
def flatten_context_list(ctxs):
    out = []
    for c in ctxs:
        if c is None:
            continue
        if isinstance(c, str):
            out.append(c)
        elif isinstance(c, list) or isinstance(c, tuple):
            out.extend(flatten_context_list(c))
        elif isinstance(c, dict):
            # if chunk dict with 'text'
            if 'text' in c and isinstance(c['text'], str):
                out.append(c['text'])
            else:
                out.append(json.dumps(c, ensure_ascii=False))
        else:
            out.append(str(c))
    return out

# ===== Self-Ask / Self-RAG / ReAct helpers (use call_generator) =====

def decompose_query(query, max_subs=3):
    system_prompt = (
        "Разбей сложный вопрос на логические подвопросы. "
        "Пиши кратко, по одному подвопросу в строку. "
        "Не пиши ответы. Если вопрос простой — просто повтори его."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query},
    ]
    text = call_generator(messages, max_new_tokens=200, do_sample=False)
    subs = [s.strip('-• \t ') for s in re.split(r'\n+', text) if s.strip()]
    if len(subs) == 0:
        return [query]
    return subs[:max_subs]

def self_rag_step(query, max_hops=3):
    reasoning_trace = []
    context_collected = []
    current_query = query

    for hop in range(max_hops):
        think_prompt = f"""You are an expert information retrieval agent.
Reasoning history:
{json.dumps(reasoning_trace, ensure_ascii=False, indent=0)}

Current question: "{current_query}"

Decide what to do next:
1) If you need more information, reply with: SEARCH: <new query>
2) If you can already answer, reply with: ANSWER: <final answer>
Respond only with one of these formats.
"""

        step_text = call_generator([{"role": "user", "content": think_prompt}], max_new_tokens=200, do_sample=False)
        reasoning_trace.append(step_text)

        if "SEARCH:" in step_text:
            sub_query = step_text.split("SEARCH:", 1)[1].strip()
            retrieved, *_ = retrieve(sub_query, top_k=15)
            reranked = rerank_candidates(sub_query, retrieved, reranker, batch_size=RERANK_BATCH)
            top_texts = flatten_context_list([r for r in reranked[:5]])
            context_collected.append("\n".join(top_texts))
            current_query = sub_query
            continue
        if "ANSWER:" in step_text:
            answer = step_text.split("ANSWER:", 1)[1].strip()
            return answer, context_collected, reasoning_trace

    return "Не удалось завершить рассуждение.", context_collected, reasoning_trace

def react_reasoning(query, max_turns=4):
    trace = []
    context = []

    for turn in range(max_turns):
        prompt = f"""
You are a reasoning agent that thinks and acts step-by-step to answer a question.
Use the following format:
Thought: <your reasoning>
Action: <SEARCH: <query> or FINISH: <final answer>>

History so far:
{json.dumps(trace, ensure_ascii=False, indent=0)}

Question: "{query}"
"""

        text = call_generator([{"role": "user", "content": prompt}], max_new_tokens=200, do_sample=False)
        trace.append(text)

        if "SEARCH:" in text:
            sub_q = text.split("SEARCH:", 1)[1].strip()
            retrieved, *_ = retrieve(sub_q, top_k=10)
            reranked = rerank_candidates(sub_q, retrieved, reranker, batch_size=RERANK_BATCH)
            obs = flatten_context_list([r for r in reranked[:3]])
            obs_text = "\n".join(obs)
            context.append(obs_text)
            trace.append(f"Observation: {obs_text[:500]}...")
            continue
        if "FINISH:" in text:
            final = text.split("FINISH:", 1)[1].strip()
            return final, context, trace

    return "Не удалось завершить рассуждение.", context, trace

# ---------- Основной цикл ----------
# ---------- Основной цикл ----------
results_out = []

for q_idx, q in enumerate(tqdm(queries, desc="Questions"), start=1):
    try:
        qid = q.get('query_id') or q.get('ID') or q.get('id')
        question = q['question']

        # 1️⃣ Self-Ask: разложение
        sub_questions = decompose_query(question, max_subs=3)
        all_contexts = []

        # 2️⃣ Для каждого подпроша — Self-RAG
        for sub_q in sub_questions:
            ans, ctxs, trace = self_rag_step(sub_q, max_hops=3)
            all_contexts.extend(ctxs)

        # 3️⃣ ReAct reasoning
        final_answer, react_contexts, react_trace = react_reasoning(question, max_turns=3)
        all_contexts.extend(react_contexts)

        # 4️⃣ Если ничего не найдено — fallback
        if len(all_contexts) == 0:
            retrieved, *_ = retrieve(question, top_k=TOP_K_RETRIEVE)
            reranked = rerank_candidates(question, retrieved, reranker, batch_size=RERANK_BATCH)
            all_contexts = flatten_context_list([r for r in reranked[:TOP_AFTER_RERANK]])

        # 5️⃣ Формируем финальный контекст
        flat_ctxs = flatten_context_list(all_contexts)
        context_concat = "\n\n".join(flat_ctxs[-TOP_AFTER_RERANK:])

        # 6️⃣ Генерация финального ответа
        prompt = f"""
You are an expert psychology assistant. Answer ONLY based on the provided textbook context.
If the information is missing, say 'No data found in the textbook.'

Question: {question}

Context:
{context_concat}

Give a concise answer in English.
Then output JSON with fields "sections" and "pages" for the cited sources.
Format example: {{"sections": ["..."], "pages": ["..."]}}
"""
        with torch.inference_mode():
            gen_text = call_generator(prompt, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)

        # 7️⃣ Извлечение JSON со ссылками
        json_refs = None
        try:
            last_brace = gen_text.rfind('{')
            if last_brace != -1:
                possible = gen_text[last_brace:]
                json_refs = json.loads(possible)
        except Exception:
            json_refs = None

        # Если модель не вывела JSON — делаем fallback по section/page
        if not json_refs:
            retrieved, *_ = retrieve(question, top_k=TOP_AFTER_RERANK)
            reranked = rerank_candidates(question, retrieved, reranker, batch_size=RERANK_BATCH)
            sections = list({r['section'] for r in reranked if r.get('section')})
            pages = list({str(r['page_no']) for r in reranked if r.get('page_no')})
            json_refs = {"sections": sections, "pages": pages}

        # 8️⃣ Убираем JSON из текста ответа
        answer_text = gen_text[:gen_text.rfind('{')].strip() if '{' in gen_text else gen_text.strip()

        # 9️⃣ Сохраняем в старом формате
        results_out.append({
            "ID": qid,
            "context": context_concat.replace('"', "'"),
            "answer": answer_text.replace('"', "'"),
            "references": json.dumps(json_refs, ensure_ascii=False)
        })

    except torch.cuda.OutOfMemoryError:
        print(f"\n⚠️ OOM on question {q_idx}: {question[:60]}...")
        torch.cuda.empty_cache()
        gc.collect()
        time.sleep(2)
        continue

    except Exception as e:
        print(f"⚠️ Error on question {q_idx}: {e}")
        continue

    if q_idx % CLEAR_EVERY == 0:
        torch.cuda.empty_cache()
        gc.collect()
        time.sleep(0.5)

print(f"\n✅ Done. Processed {len(results_out)} questions.")


Questions:   0%|          | 0/50 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'


✅ Done. Processed 50 questions.


In [41]:
df = pd.DataFrame(results_out)[['ID','context','answer','references']]
df.to_csv('submission.csv', index=False, quoting=csv.QUOTE_MINIMAL)

In [42]:
df

,ID,context,answer,references
0,1,42 2 • Psychological Research\nexplain behavio...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""51"", ""4..."
1,2,"yelin sheath, which increases the speed of\ntr...",You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""96"", ""9..."
2,3,articular emphasis on sleep. The different sta...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""121"", ""..."
3,4,2).\nPositive and Negative Reinforcement and P...,You are an expert psychology assistant. Answer...,"{""sections"": [], ""pages"": []}"
4,5,ed to identify the problem and then apply a\ns...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5"", ""Industrial-Organiz..."
5,6,250 8 • Memory\nattempt to memorize the concep...,You are an expert psychology assistant. Answer...,"{""sections"": [], ""pages"": []}"
6,7,include problems\nin using and understanding ...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""226"", ""..."
7,8,oximation of the basic personality dimensions ...,You are an expert psychology assistant. Answer...,"{""sections"": [""Preface 5""], ""pages"": [""396"", ""..."
8,9,12 • Summary 439\nSummary\n12.1 What Is Social...,You are an expert psychology assistant. Answer...,"{""sections"": [], ""pages"": []}"
9,10,d\naddress both problems simultaneously.\n16.5...,You are an expert psychology assistant. Answer...,"{""sections"": [], ""pages"": []}"
